# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data, check for leakage, evaluate with a **time-ordered** split (we would deploy on future inspections), compare against baselines.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'assets:', df.asset_tag.nunique(), '| missing:', df.isna().sum().sum(), '| range:', df.inspected_at.min(), '->', df.inspected_at.max())
print('base rate:', df.failed_90d.mean().round(4))
df.describe().T

## 1. Leakage check: `parts_replaced`
The dictionary says it is the parts replaced on the work order that *closed* this inspection record, so it is probably recorded after the outcome (repairs following a failure). It would not be available at inspection time.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

Every record with 2 or more parts replaced failed, and none with 0 did. That is the outcome leaking through, not a usable signal. **It is excluded from all models.** Each `asset_tag` appears once, so there is no repeated-asset leakage.

## 2. Failure rate over time

In [ ]:
m = df.groupby(df.inspected_at.dt.to_period('M')).failed_90d.agg(['mean', 'size'])
ax = m['mean'].plot(marker='o', figsize=(9, 3), title='Failure rate by inspection month')
ax.axhline(df.failed_90d.mean(), ls='--', c='grey'); plt.show()

The base rate swings from about 0.3% to 39% by month (May-Aug 2024 and Jan 2025 are high). Causes could be seasonality, a real operating regime, or labeling effects. The latest months may also be right-censored if failures after the data cut-off are not recorded; the data cannot tell us. Two years of data cannot separate seasonality from one-off events. This matters for evaluation: a random split lets a model learn month-specific rates from neighbouring rows, which will not hold for future inspections.

## 3. Features vs. outcome

In [ ]:
feats = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
print(df[feats + ['failed_90d']].corr()['failed_90d'].drop('failed_90d').round(3))
print(df.groupby('site').failed_90d.mean().round(3))
fig, axs = plt.subplots(1, 5, figsize=(16, 3))
for a, f in zip(axs, feats):
    df.groupby(pd.qcut(df[f], 10, duplicates='drop'), observed=True).failed_90d.mean().reset_index(drop=True).plot(ax=a, marker='o', title=f)
    a.set_ylim(0, None)
plt.tight_layout(); plt.show()

## 4. Models
Features: the five measurements plus `site` (one-hot). No `parts_replaced`. Date and month are **not** used as features, since two years cannot support a seasonality claim.
Two evaluations:
- **Random 5-fold CV**: optimistic reference.
- **Forward-chaining time split**: train on everything before a quarter, test on that quarter, for each quarter from 2024Q3 on. This mimics real use.

In [ ]:
X = pd.get_dummies(df[feats + ['site']], columns=['site'], dtype=float)
y = df.failed_90d.values
models = {'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          'hgb': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)}
def rep(name, yt, p):
    return dict(eval=name, auc=roc_auc_score(yt, p), pr_auc=average_precision_score(yt, p), base=yt.mean())
rows, fold = [], []
skf = StratifiedKFold(5, shuffle=True, random_state=0)
for n, mdl in models.items():
    p = cross_val_predict(mdl, X, y, cv=skf, method='predict_proba')[:, 1]
    rows.append({'model': n, **rep('random CV', y, p)})
q = df.inspected_at.dt.to_period('Q')
tests = sorted(q.unique())[2:]
for n, mdl in models.items():
    ps, ys = [], []
    for t in tests:
        tr, te = (q < t).values, (q == t).values
        mdl.fit(X[tr], y[tr]); p = mdl.predict_proba(X[te])[:, 1]
        ps.append(p); ys.append(y[te])
        fold.append(dict(model=n, test=str(t), n=te.sum(), base=y[te].mean(),
                         auc=roc_auc_score(y[te], p), pr_auc=average_precision_score(y[te], p)))
    rows.append({'model': n, **rep('forward-chaining (pooled)', np.concatenate(ys), np.concatenate(ps))})
print(pd.DataFrame(rows).round(3).to_string(index=False)); print()
print(pd.DataFrame(fold).round(3).to_string(index=False))

## 5. Coefficients (logistic regression, standardized features)

In [ ]:
lr = models['logreg'].fit(X, y)
print(pd.Series(lr[-1].coef_[0], X.columns).sort_values().round(3))

## Conclusions
- **Yes, moderately.** On genuinely future quarters, gradient boosting reaches AUC about 0.73-0.84 per quarter (pooled 0.78), against 0.5 for chance. PR-AUC is 0.12-0.49 versus base rates of 3-27%, so it ranks risk well but is far from a precise alarm.
- **Random CV overstates performance** (HGB AUC 0.85 vs 0.78 forward-chaining), most likely because rows from the same month share a base rate. Quote the forward-chaining numbers.
- **`parts_replaced` is leakage and excluded.** Including it would give a near-perfect but useless model.
- Higher vibration, load and bearing temperature, and lower meter hours, are associated with failure. `site` carries no signal. HGB beating logistic regression suggests non-linear effects or interactions.
- **Caveats:** the monthly failure rate varies wildly (0.3% to 39%) for reasons the data cannot explain, so absolute probabilities would not be calibrated for the future. The latest months may be right-censored. Next steps: ask what drove the regime shifts, confirm whether post-cut-off failures are recorded, then calibrate and pick an alert threshold from inspection capacity and failure cost.